# Traitement automatisé — toutes les visites (V0, V1, V3, V5, Vc)



In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from functools import reduce
import re

In [4]:
def construire_tableau_synthetique(result, v):
    suffix = f"_V{v}"
    dfs = []

    # 1. Feuille Vx
    df_v = result[f"V{v}"].copy()
    df_v = df_v.drop(columns=[c for c in df_v.columns if c.upper() in ("VISITE", "VISIT")], errors="ignore")
    df_v = df_v.rename(columns={c: f"{c}{suffix}" for c in df_v.columns if c != "SUBJID"})
    dfs.append(df_v)

    # 2. PDQ39_SI uniquement
    if "PDQ39" in result:
        df = result["PDQ39"][["SUBJID", "PDQ39_SI"]].copy()
        df = df.rename(columns={"PDQ39_SI": f"PDQ39_SI{suffix}"})
        dfs.append(df)

    # 3. UPDRSIII_S pour V0, sinon UPDRSIII
    key = "UPDRSIII" if (v == 0 and "UPDRSIII" in result) else "UPDRSIII"
    if key in result:
        df = result[key].copy()
        df = df.drop(columns=[c for c in df.columns if c.upper() in ("VISITE", "VISIT")], errors="ignore")
        # if key =="UPDRSIII" and v != 1 :
        #     df=df.drop(columns=["UPDRSIII_tot","statut"])
        df = df.rename(columns={c: f"{c}{suffix}" for c in df.columns if c != "SUBJID"})
        dfs.append(df)

    # 4. UPDRSIV
    if "UPDRSIV" in result:
        df = result["UPDRSIV"].copy()
        df = df.drop(columns=[c for c in df.columns if c.upper() in ("VISITE", "VISIT")], errors="ignore")
        df = df.rename(columns={c: f"{c}{suffix}" for c in df.columns if c != "SUBJID"})
        dfs.append(df)

    # 5. ECMP
    if "ECMP" in result:
        df = result["ECMP"].copy()
        df = df.drop(columns=[c for c in df.columns if c.upper() in ("VISITE", "VISIT")], errors="ignore")
        df = df.rename(columns={c: f"{c}{suffix}" for c in df.columns if c != "SUBJID"})
        dfs.append(df)

    # 6. QUIP
    if "QUIP" in result:
        df = result["QUIP"].copy()
        df = df.drop(columns=[c for c in df.columns if c.upper() in ("VISITE", "VISIT")], errors="ignore")
        df = df.rename(columns={c: f"{c}{suffix}" for c in df.columns if c != "SUBJID"})
        dfs.append(df)

    # 7. LARS
    if "LARS" in result:
        df = result["LARS"].copy()
        df = df.drop(columns=[c for c in df.columns if c.upper() in ("VISITE", "VISIT","LARS_SCORE","statut","LARS_RESULTAT")], errors="ignore")
        df = df.rename(columns={c: f"{c}{suffix}" for c in df.columns if c != "SUBJID"})
        dfs.append(df)

    # 8. UPPS
    if "UPPS" in result:
        df = result["UPPS"].copy()
        df = df.drop(columns=[c for c in df.columns if c.upper() in ("VISITE", "VISIT")], errors="ignore")
        df = df.rename(columns={c: f"{c}{suffix}" for c in df.columns if c != "SUBJID"})
        dfs.append(df)

    # 9. DIGITSMT_TRAILMT_DKEFS
    if "DIGITSMT_TRAILMT_DKEFS" in result:
        df = result["DIGITSMT_TRAILMT_DKEFS"].copy()
        df = df.drop(columns=[c for c in df.columns if c.upper() in ("VISITE", "VISIT")], errors="ignore")
        df = df.rename(columns={c: f"{c}{suffix}" for c in df.columns if c != "SUBJID"})
        dfs.append(df)

    # 10. HAMD_tot uniquement
    if "HAMD" in result:
        df = result["HAMD"][["SUBJID", "HAMD_tot"]].copy()
        df = df.rename(columns={"HAMD_tot": f"HAMD_tot{suffix}"})
        dfs.append(df)

    # 11. HAMA_tot uniquement
    if "HAMA" in result:
        df = result["HAMA"][["SUBJID", "HAMA_tot"]].copy()
        df = df.rename(columns={"HAMA_tot": f"HAMA_tot{suffix}"})
        dfs.append(df)

    # 12. MOCA_tot uniquement
    if "MOCA" in result:
        df = result["MOCA"][["SUBJID", "MOCA_tot"]].copy()
        df = df.rename(columns={"MOCA_tot": f"MOCA_tot{suffix}"})
        dfs.append(df)

    # 13. PSYCHOTROPES → colonnes CLASS* uniquement
    if "PSYCHOTROPES" in result:
        df = result["PSYCHOTROPES"][["SUBJID",'Anxiolytiques', 'Antidépresseurs','Neuroleptiques', 'Thymorégulateurs']].copy()
        df = df.rename(columns={
        'Anxiolytiques': f"Anxiolytiques{suffix}",
        'Antidépresseurs': f"Antidépresseurs{suffix}",
        'Neuroleptiques': f"Neuroleptiques{suffix}",
        'Thymorégulateurs': f"Thymorégulateurs{suffix}"
        })
        dfs.append(df)
        

    # 14. AUTRE_PARKINSON → colonnes CLASS* uniquement
    # if "AUTRE_PARKINSON" in result:
    #     df = result["AUTRE_PARKINSON"].copy()
    #     cols = ["SUBJID"] + [c for c in df.columns if c.upper().startswith("CLASS")]
    #     df = df[cols]
    #     df = df.rename(columns={c: f"{c}{suffix}" for c in df.columns if c != "SUBJID"})
    #     dfs.append(df)

    # 15. LEDD
    if "LEDD" in result:
        df = result["LEDD"].copy()
        df = df.drop(columns=[c for c in df.columns if c.upper() in ("VISITE", "VISIT")], errors="ignore")
        df = df.rename(columns={c: f"{c}{suffix}" for c in df.columns if c != "SUBJID"})
        dfs.append(df)
    
    if "FREQUENCE" in result :
        df = result["FREQUENCE"].copy()
        df = df.drop(columns=[c for c in df.columns if c.upper() in ("VISITE")], errors="ignore")
        df = df.rename(columns={c: f"{c}{suffix}" for c in df.columns if c != "SUBJID"})
        dfs.append(df)

    # ── Fusion sur SUBJID ─────────────────────────────────────────────
    df_final = dfs[0]
    for df_next in dfs[1:]:
        df_final = pd.merge(df_final, df_next, on="SUBJID", how="outer")

    return df_final

---
## Fonction principale : `traiter_visite(v, vc=False)`

In [21]:
def traiter_visite(v, vc=False):

    chemin = f"Output/version_4/V{v}.xlsx"
    print(f"\n{'='*60}")
    print(f"  TRAITEMENT  V{v}  —  {chemin}")
    print(f"{'='*60}")

    # ==================================================================
    # FEUILLES COMMUNES  (présentes dans toutes les visites, y compris Vc)
    # ==================================================================
     
    df_V = pd.read_excel(chemin, sheet_name=f"V{v}")
    # df_V = df_V.iloc[1:].reset_index(drop=True)
    # df_V = nettoyer_codes_manquants(df_V)

    # ── LEDD ──────────────────────────────────────────────────────────
    df_LEDD = pd.read_excel(chemin, sheet_name="LEDD")
    # df_LEDD["somme_calc"] = df_LEDD.iloc[:, 2:-1].sum(axis=1, skipna=True)
    # df_LEDD["statut"] = np.where(
    #     np.isclose(df_LEDD["somme_calc"], df_LEDD["ledd_tot"], atol=0.01),
    #     "ok", "différent"
    # )
    # df_LEDD["ledd_tot"] = df_LEDD["somme_calc"]
    # df_LEDD.drop(columns=["somme_calc", "statut"], inplace=True)
    # df_Total_LEDD = df_LEDD[["SUBJID", "ledd_tot"]]
    print(f"df_LEDD                 : {df_LEDD.shape}")

    # ── PSYCHOTROPES ──────────────────────────────────────────────────
    df_PSYCHOTROPES = pd.read_excel(chemin, sheet_name="PSYCHOTROPES")
    # df_PSYCHOTROPES = df_PSYCHOTROPES[["VISITE","SUBJID","Anxiolytiques","Antidépresseurs","Neuroleptiques","Thymorégulateurs"]]
    # print(f"\nFeuille PSYCHOTROPES    : {df_PSYCHOTROPES.shape}")
    # df_PSYCHOTROPES = nettoyer_codes_manquants(df_PSYCHOTROPES)
    # cols = df_PSYCHOTROPES.columns[:2].tolist()
    # cols += [c for c in df_PSYCHOTROPES.columns if c.startswith('MEDICMT') or c.startswith('POSO')]
    # df_PSYCHOTROPES = df_PSYCHOTROPES[cols]
    # df_PSYCHOTROPES = classer_medicaments(df_PSYCHOTROPES, dict_complet)
    print(f"df_PSYCHOTROPES         : {df_PSYCHOTROPES.shape}")

    # ── AUTRE_PARKINSON ───────────────────────────────────────────────
    # df_AUTRE_PARKINSON = pd.read_excel(chemin, sheet_name="AUTRE_PARKINSON")
    # # df_AUTRE_PARKINSON = nettoyer_codes_manquants(df_AUTRE_PARKINSON)
    # # cols = df_AUTRE_PARKINSON.columns[:2].tolist()
    # # cols += [c for c in df_AUTRE_PARKINSON.columns if c.startswith('MEDICMT') or c.startswith('POSO')]
    # # df_AUTRE_PARKINSON = df_AUTRE_PARKINSON[cols]
    # # df_AUTRE_PARKINSON = classer_medicaments(df_AUTRE_PARKINSON, dict_complet)
    # print(f"df_AUTRE_PARKINSON      : {df_AUTRE_PARKINSON.shape}")

    # ── CONSO_SPECIFIQUE ──────────────────────────────────────────────
    # df_CONSO_SPECIFIQUE = pd.read_excel(chemin, sheet_name="CONSO_SPECIFIQUE")
    # # df_CONSO_SPECIFIQUE = nettoyer_codes_manquants(df_CONSO_SPECIFIQUE)
    # # cols = df_CONSO_SPECIFIQUE.columns[:2].tolist()
    # # cols += [c for c in df_CONSO_SPECIFIQUE.columns if c.startswith('MEDICMT') or c.startswith('POSO')]
    # # df_CONSO_SPECIFIQUE = df_CONSO_SPECIFIQUE[cols]
    # print(f"df_CONSO_SPECIFIQUE     : {df_CONSO_SPECIFIQUE.shape}")

    # ==================================================================
    # CAS SPÉCIAL  →  Vc  (uniquement les feuilles communes)
    # ==================================================================
    # if vc:
    #     return {
    #         f"V{v}"             : df_V,
    #         "LEDD"             : df_LEDD,
    #         "CONSO_SPECIFIQUE" : df_CONSO_SPECIFIQUE,
    #         "PSYCHOTROPES"     : df_PSYCHOTROPES,
    #         "AUTRE_PARKINSON"  : df_AUTRE_PARKINSON,
    #     }
        
    if vc:
        result = {
            f"V{v}"             : df_V,
            "LEDD"             : df_LEDD,
            # "CONSO_SPECIFIQUE" : df_CONSO_SPECIFIQUE,
            "PSYCHOTROPES"     : df_PSYCHOTROPES,
            # "AUTRE_PARKINSON"  : df_AUTRE_PARKINSON,
        }

        result["SYNTHESE"] = construire_tableau_synthetique(result, v)

        return result
    # ==================================================================
    # FEUILLES SPÉCIFIQUES  V0 → V5
    # ==================================================================

    # ── UPDRSIII ──────────────────────────────────────────────────────
    if v == 0:
        df_UPDRSIII = pd.read_excel(chemin, sheet_name="UPDRSIII")
        df_UPDRSIII = df_UPDRSIII[["VISITE","SUBJID","UPDRSIII_SCORE_TOTAL_OFF","UPDRSIII_BEST_ON","UPDRSIII_DOPASENSIBILITE_PCT"]] 
        print(f"\nFeuille UPDRSIII        : {df_UPDRSIII.shape}")

    elif v == 1:
        df_UPDRSIII = pd.read_excel(chemin, sheet_name="UPDRSIII")
        # df_UPDRSIII = df_UPDRSIII.iloc[1:].reset_index(drop=True)

        # df_UPDRSIII = df_UPDRSIII.rename(columns={c: f"UPDRSIII_{c}" for c in df_UPDRSIII.columns if c not in ("SUBJID", "VISITE")}) 
        df_UPDRSIII = df_UPDRSIII[["VISITE","SUBJID","UPDRSIII_OFF_TOTALCALC","UPDRSIII_ON_TOTALCALC"]]
        # df_UPDRSIII = df_UPDRSIII[["VISITE","SUBJID","OFF_TOTALCALC","ON_TOTALCALC"]]

    elif v in (3, 5):
        df_UPDRSIII = pd.read_excel(chemin, sheet_name="UPDRSIII")
        # df_UPDRSIII = df_UPDRSIII.iloc[:,:-2]

        # df_UPDRSIII = df_UPDRSIII.rename(columns={c: f"UPDRSIII_{c}" for c in df_UPDRSIII.columns if c not in ("SUBJID", "VISITE")}) 
        df_UPDRSIII = df_UPDRSIII[["VISITE","SUBJID","UPDRSIII_ON_TOTAL"]]
        print(f"\nFeuille UPDRSIII        : {df_UPDRSIII.shape}")

    # if v == 0:
    #     df_UPDRSIII = pd.read_excel(chemin, sheet_name="UPDRSIII_S")
    #     df_UPDRSIII = df_UPDRSIII.iloc[1:].reset_index(drop=True)
    #     print(f"\nFeuille UPDRSIII        : {df_UPDRSIII.shape}")
    # elif v == 1:
    #     df_UPDRSIII = pd.read_excel(chemin, sheet_name="UPDRSIII")
    #     df_UPDRSIII = df_UPDRSIII.iloc[1:].reset_index(drop=True)
    #     # df_UPDRSIII = df_UPDRSIII[["VISITE","SUBJID","OFF_TOTALCALC","ON_TOTALCALC"]]
    # elif v in (3, 5):
    #     df_UPDRSIII = pd.read_excel(chemin, sheet_name="UPDRSIII")
    #     df_UPDRSIII = df_UPDRSIII.iloc[:,:-2]
    #     print(f"\nFeuille UPDRSIII        : {df_UPDRSIII.shape}")
        

    # ── UPDRSIV ───────────────────────────────────────────────────────
    df_UPDRSIV = pd.read_excel(chemin, sheet_name="UPDRSIV")
    # df_UPDRSIV.drop("MDSUPDRSPARTIE4",axis=1,inplace=True)
    # df_UPDRSIV.drop("UPDRSIV_tot",axis=1)
    df_UPDRSIV = df_UPDRSIV[["VISITE","SUBJID","UPDRSIV_tot"]]
    
    # ── PDQ39 ─────────────────────────────────────────────────────────
    df_PDQ39 = pd.read_excel(chemin, sheet_name="PDQ39")
    df_PDQ39 = df_PDQ39[["VISITE","SUBJID","PDQ39_SI"]]
    print(f"\nFeuille PDQ39           : {df_PDQ39.shape}")

    # ── QUIP ──────────────────────────────────────────────────────────
    df_QUIP = pd.read_excel(chemin, sheet_name="QUIP")
    print(f"\nFeuille QUIP            : {df_QUIP.shape}")


    # ── MOCA ──────────────────────────────────────────────────────────
    df_MOCA = pd.read_excel(chemin, sheet_name="MOCA")
    df_MOCA = df_MOCA[["VISITE","SUBJID","MOCA_tot"]]
    print(f"\nFeuille MOCA            : {df_MOCA.shape}")


    # ── HAMA ──────────────────────────────────────────────────────────
    df_HAMA = pd.read_excel(chemin, sheet_name="HAMA")
    df_HAMA = df_HAMA[["VISITE","SUBJID","HAMA_tot"]]
    print(f"df_HAMA                 : {df_HAMA.shape}")

    # ── HAMD ──────────────────────────────────────────────────────────
    df_HAMD = pd.read_excel(chemin, sheet_name="HAMD")
    df_HAMD = df_HAMD[["VISITE","SUBJID", "HAMD_tot"]]
    print(f"df_HAMD                 : {df_HAMD.shape}")

    # ── LARS ──────────────────────────────────────────────────────────
    df_LARS = pd.read_excel(chemin, sheet_name="LARS")
    # df_LARS.drop(columns=["LARS_tot","statut"],inplace=True)
    # df_LARS = df_LARS.iloc[:, :-3]
    df_LARS = df_LARS[["VISITE","SUBJID","LARS_IC","LARS_E","LARS_AI","LARS_SA","LARS_SCORE"]]
    print(f"df_LARS                 : {df_LARS.shape}")

    # ── ECMP ──────────────────────────────────────────────────────────
    df_ECMP = pd.read_excel(chemin, sheet_name="ECMP")
    cols_atcd = [col for col in df_ECMP.columns if col.endswith('ATCD')]
    df_ECMP = df_ECMP.drop(columns=cols_atcd)
    print(f"\nFeuille ECMP            : {df_ECMP.shape}")
    

    # ── DIGITSMT_TRAILMT_DKEFS ────────────────────────────────────────
    df_DIGITSMT = pd.read_excel(chemin, sheet_name="DIGITSMT_TRAILMT_DKEFS")
    print(f"\nFeuille DIGITSMT        : {df_DIGITSMT.shape}")


    # ── UPPS (fichier externe, V0 V1 V3 V5) ──────────────────────────
    if v in (0, 1, 3, 5):
        df_UPPS = pd.read_excel(chemin, sheet_name="UPPS")
        print(f"df_UPPS                 : {df_UPPS.shape}")

    # ── FREQUENCE  (V1, V2, V3) ───────────────────────────────────────
    # if v in (1, 3, 5):
    #     df_FREQUENCE = pd.read_excel(chemin, sheet_name="FREQUENCE")



    # ==================================================================
    # CONSTRUCTION DU DICTIONNAIRE DE RETOUR
    # ==================================================================
    result = {
        f"V{v}"                   : df_V,
        "LEDD"                    : df_LEDD,
        # "CONSO_SPECIFIQUE"        : df_CONSO_SPECIFIQUE,
        "PSYCHOTROPES"            : df_PSYCHOTROPES,
        # "AUTRE_PARKINSON"         : df_AUTRE_PARKINSON,
        "UPDRSIII"                : df_UPDRSIII,
        "UPDRSIV"                 : df_UPDRSIV,
        "PDQ39"                   : df_PDQ39,
        "QUIP"                    : df_QUIP,
        "MOCA"                    : df_MOCA,
        "HAMA"                    : df_HAMA,
        "HAMD"                    : df_HAMD,
        "LARS"                    : df_LARS,
        "ECMP"                    : df_ECMP,
        "DIGITSMT_TRAILMT_DKEFS"  : df_DIGITSMT,
    }



    if v in (0, 1, 3, 5):
        result["UPPS"] = df_UPPS

    # if v in (1, 3, 5):
    #     result["FREQUENCE"] = df_FREQUENCE
    
    # # ── SYNTHESE ──────────────────────────────────────────────────────
    # result["SYNTHESE"] = construire_tableau_synthetique(result, v)


    return result

In [12]:
ORDRE_FEUILLES = [
    
    "PDQ39",
    "UPDRSIII",
    "UPDRSIV",  
    "ECMP",
    "QUIP",
    "LARS",
    "UPPS",
    "HAMD",
    "HAMA",
    "MOCA",
    "DIGITSMT_TRAILMT_DKEFS",
    # "CONSO_SPECIFIQUE",
    "PSYCHOTROPES",
    # "AUTRE_PARKINSON",
    "LEDD",
    "FREQUENCE"
    # "SYNTHESE"
]

def reordonner_feuilles(sheets, v):
    cle_visite    = f"V{v}"
    ordre_complet = [cle_visite] + ORDRE_FEUILLES
    return {k: sheets[k] for k in ordre_complet if k in sheets}

---
## Utilitaire d'écriture Excel

In [9]:
def ecrire_excel(sheets_dict, version, output_dir="Output/Statistiques"):
    filepath = f"{output_dir}/V{version}.xlsx"
    with pd.ExcelWriter(filepath, engine="openpyxl") as writer:
        for sheet_name, df in sheets_dict.items():
            df.to_excel(writer, sheet_name=sheet_name[:31], index=False)
    print(f"[OK] {filepath}  →  {len(sheets_dict)} feuilles")

---
## Traitement + écriture de chaque visite

### Vc

In [16]:
sheets_Vc = traiter_visite("c", vc=True)
sheets_Vc = reordonner_feuilles(sheets_Vc, "c")
ecrire_excel(sheets_Vc, "c")


  TRAITEMENT  Vc  —  Output/version_4/Vc.xlsx
df_LEDD                 : (491, 7)
df_PSYCHOTROPES         : (835, 6)
[OK] Output/Statistiques/Vc.xlsx  →  3 feuilles


### V0

In [19]:
sheets_V0 = traiter_visite(0)
sheets_V0 = reordonner_feuilles(sheets_V0, 0)

ecrire_excel(sheets_V0, 0)



  TRAITEMENT  V0  —  Output/version_4/V0.xlsx
df_LEDD                 : (794, 7)
df_PSYCHOTROPES         : (835, 6)

Feuille UPDRSIII        : (835, 5)

Feuille PDQ39           : (835, 3)

Feuille QUIP            : (835, 33)

Feuille MOCA            : (835, 3)
df_HAMA                 : (835, 3)
df_HAMD                 : (835, 3)
df_LARS                 : (835, 7)

Feuille ECMP            : (835, 23)

Feuille DIGITSMT        : (835, 21)
df_UPPS                 : (835, 22)
[OK] Output/Statistiques/V0.xlsx  →  14 feuilles


### V1

In [22]:
sheets_V1 = traiter_visite(1)
sheets_V1 = reordonner_feuilles(sheets_V1, 1)
ecrire_excel(sheets_V1, 1)


  TRAITEMENT  V1  —  Output/version_4/V1.xlsx
df_LEDD                 : (525, 7)
df_PSYCHOTROPES         : (835, 6)

Feuille PDQ39           : (835, 3)

Feuille QUIP            : (835, 33)

Feuille MOCA            : (835, 3)
df_HAMA                 : (835, 3)
df_HAMD                 : (835, 3)
df_LARS                 : (835, 7)

Feuille ECMP            : (835, 23)

Feuille DIGITSMT        : (835, 21)
df_UPPS                 : (835, 22)
[OK] Output/Statistiques/V1.xlsx  →  14 feuilles


### V3

In [23]:
sheets_V3 = traiter_visite(3)
sheets_V3 = reordonner_feuilles(sheets_V3, 3)
ecrire_excel(sheets_V3, 3)


  TRAITEMENT  V3  —  Output/version_4/V3.xlsx
df_LEDD                 : (272, 7)
df_PSYCHOTROPES         : (835, 6)

Feuille UPDRSIII        : (835, 3)

Feuille PDQ39           : (835, 3)

Feuille QUIP            : (835, 33)

Feuille MOCA            : (835, 3)
df_HAMA                 : (835, 3)
df_HAMD                 : (835, 3)
df_LARS                 : (835, 7)

Feuille ECMP            : (835, 23)

Feuille DIGITSMT        : (835, 21)
df_UPPS                 : (835, 22)
[OK] Output/Statistiques/V3.xlsx  →  14 feuilles


### V5

In [24]:
sheets_V5 = traiter_visite(5)
sheets_V5 = reordonner_feuilles(sheets_V5, 5)
ecrire_excel(sheets_V5, 5)


  TRAITEMENT  V5  —  Output/version_4/V5.xlsx
df_LEDD                 : (313, 7)
df_PSYCHOTROPES         : (835, 6)

Feuille UPDRSIII        : (835, 3)

Feuille PDQ39           : (835, 3)

Feuille QUIP            : (835, 33)

Feuille MOCA            : (835, 3)
df_HAMA                 : (835, 3)
df_HAMD                 : (835, 3)
df_LARS                 : (835, 7)

Feuille ECMP            : (835, 23)

Feuille DIGITSMT        : (835, 21)
df_UPPS                 : (835, 22)
[OK] Output/Statistiques/V5.xlsx  →  14 feuilles


#

# Prétraitement info statiques 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [25]:
info =  pd.read_excel("Output/version_3/info.xlsx")
info.head()


,SUBJID,INIT_PAT,D_1ER_SYMPT,D_DIAG,D_LDOPA,D_TTT_DOPAM,D_FLUCTU_MOTR,D_FLUCTU_NONMOTR,D_DYSKINESIE,Mut_GBA,...,AGE,SEXE,SUIVI_ETUDE,D_FIN_ETUDE,D_SORTIE_PREMA,MOTIF_SORTIE_PREMA,AUTRE_PRECIS,D_INVESTIGATEUR,NOM_INVESTIGATEUR,MOTIF_SORTIE_PREMA1
0,Subject Identifier for the Study,initiales patient,année premiers symptomes,année du diagnostic de la maladie,année d'introduction de la L-DOPA,année d'introduction du traitement dopaminergique,année d'apparition des fluctuations motrices,année d'apparition des fluctuations non motrices,année d'apparition des dyskinésies,NaN,...,âge,sexe,suivi étude,date de fin de l'étude,date de sortie prématurée,motif de sortie prématurée,autre précision,date de signature de l'investigateur,nom de l'investigateur,motif de sortie prématurée LIB
1,01-001,SR,1998,1999,2000,2000,2002,2000,NaN,NaN,...,67,1,0,NaN,22/11/2013,1,NaN,18/03/2014,MOREAU CAROLINE,Patient non opéré
2,01-002,TM,2006,2006,2006,2006,2011,NaN,NaN,NaN,...,59,1,0,NaN,20/02/2014,6,SUSPICION DE CANCER PULMONAIRE,24/03/2014,DR MOREAU,Autre
3,01-003,SJ,2001,2001,2003,2001,2004,NaN,2004,NaN,...,61,1,0,NaN,07/03/2014,1,NaN,21/03/2014,DR HOPES LUCIE,Patient non opéré
4,01-004,DJ,1998,2000,2003,2000,2003,2003,2003,NaN,...,65,2,0,NaN,09/01/2015,6,RETRAIT DU MATERIEL ET RETRAIT DE CONSENTEMENT,09/01/2015,DEVOS,Autre


# Fusionner 

In [30]:
def reordonner_colonnes_par_variable(df):
    """
    Réorganise les colonnes du DataFrame en groupant chaque variable
    sur toutes ses visites.

    Cas spéciaux :
    - Les colonnes UPDRSIII_* sont groupées par visite (V0 ensemble, V1 ensemble…)
      car le nom de sous-colonne diffère d'une visite à l'autre.
    - Les colonnes qui n'existent qu'en V1/V3/V5 (absentes de V0 et VC)
      sont placées à la fin, groupées par variable.
    """
    from collections import defaultdict

    VISITES = ["V0", "Vc", "V1", "V3", "V5"]
    PREFIXES_PAR_VISITE = ["UPDRSIII"]  # ajouter d'autres préfixes ici si besoin

    autres_cols = [c for c in df.columns if c != "SUBJID"]

    def extraire_base_et_visite(col):
        for v in sorted(VISITES, key=len, reverse=True):
            if col.endswith(f"_{v}"):
                base = col[:-(len(v) + 1)]
                return base, v
        return col, None

    def est_prefixe_special(base):
        return any(base == p or base.startswith(f"{p}_") for p in PREFIXES_PAR_VISITE)

    # -- Classifier chaque colonne --
    groupes_normaux  = defaultdict(dict)   # base → {visite: colonne}
    groupes_speciaux = defaultdict(list)   # visite → [colonnes UPDRSIII]
    sans_visite      = []

    for col in autres_cols:
        base, visite = extraire_base_et_visite(col)
        if visite is None:
            sans_visite.append(col)
        elif est_prefixe_special(base):
            groupes_speciaux[visite].append(col)
        else:
            groupes_normaux[base][visite] = col

    # -- Séparer colonnes présentes en V0/VC vs absentes --
    visites_principales = {"V0", "Vc"}

    bases_principales = []   # présentes dans au moins V0 ou VC
    bases_secondaires = []   # absentes de V0 et VC (ex: uniquement V1/V3/V5)

    ordre_bases_vu = list(dict.fromkeys(
        extraire_base_et_visite(c)[0]
        for c in autres_cols
        if extraire_base_et_visite(c)[1] is not None
        and not est_prefixe_special(extraire_base_et_visite(c)[0])
    ))

    for base in ordre_bases_vu:
        visites_dispo = set(groupes_normaux[base].keys())
        if visites_dispo & visites_principales:
            bases_principales.append(base)
        else:
            bases_secondaires.append(base)

    # -- Construire l'ordre final --
    colonnes_ordonnees = ["SUBJID"]

    # 1. Variables normales présentes dès V0/VC — groupées par variable
    for base in bases_principales:
        for v in VISITES:
            if v in groupes_normaux[base]:
                colonnes_ordonnees.append(groupes_normaux[base][v])

    # 2. Colonnes UPDRSIII — groupées par visite (toutes V0, puis toutes V1…)
    for v in VISITES:
        colonnes_ordonnees.extend(groupes_speciaux.get(v, []))

    # 3. Variables normales absentes de V0/VC — groupées par variable, à la fin
    for base in bases_secondaires:
        for v in VISITES:
            if v in groupes_normaux[base]:
                colonnes_ordonnees.append(groupes_normaux[base][v])

    # 4. Colonnes sans suffixe visite
    colonnes_ordonnees += sans_visite

    # Garder uniquement les colonnes qui existent vraiment
    colonnes_ordonnees = [c for c in colonnes_ordonnees if c in df.columns]

    return df[colonnes_ordonnees]

In [31]:
def construire_df_final(dossier="."):
    """
    Lit directement les fichiers Excel de chaque visite,
    construit le tableau synthétique pour chaque visite,
    merge tout et réorganise les colonnes.
    """
    visites = [
        ("v0.xlsx", 0),
        ("vc.xlsx", "c"),
        ("v1.xlsx", 1),
        ("v3.xlsx", 3),
        ("v5.xlsx", 5),
    ]

    dfs = []
    for fichier, v in visites:
        chemin = os.path.join(dossier, fichier)
        if not os.path.exists(chemin):
            print(f"Fichier manquant : {fichier}, ignoré.")
            continue
        
        # Lire toutes les feuilles d'un coup → dict {nom_feuille: DataFrame}
        result = pd.read_excel(chemin, sheet_name=None)
        
        df_v = construire_tableau_synthetique(result, v)
        dfs.append(df_v)

    # Merge tous les Vx sur SUBJID
    from functools import reduce
    df_all = reduce(lambda l, r: pd.merge(l, r, on="SUBJID", how="outer"), dfs)

    # Réorganiser les colonnes par variable
    df_final = reordonner_colonnes_par_variable(df_all)

    return df_final

In [33]:
import os

df_final = construire_df_final(dossier="Output/version_4")
df_final.to_excel("Output/Statistiques/data.xlsx", index=False)

In [34]:
df_info = pd.read_excel("Output/version_3/info.xlsx")
df_info = df_info.iloc[1:]
df_complet = pd.merge(df_info, df_final, on="SUBJID", how="outer")

# Mettre SUBJID en premier
cols = ["SUBJID"] + [c for c in df_complet.columns if c != "SUBJID"]
df_complet = df_complet[cols]

df_complet.to_excel("Output/Statistiques/data.xlsx", index=False)

In [ ]:
# def reordonner_colonnes_par_variable(df, 
#                                       prefixes_updrs=None,
#                                       prefixes_frequence=None):
#     """
#     Réorganise les colonnes du DataFrame fusionné.

#     Règles :
#     - Colonnes normales  → groupées par variable : DATE_V0, DATE_V1, ... POIDS_V0, POIDS_V1, ...
#     - Colonnes UPDRSIII  → bloc complet par visite : [toutes cols V0], [toutes cols V1], ...
#     - Colonnes FREQUENCE → idem, placées TOUT à la fin

#     Parameters
#     ----------
#     df : DataFrame produit par construire_df_final / df_all
#     prefixes_updrs : liste de préfixes identifiant les colonnes UPDRSIII.
#         Par défaut, les préfixes détectés automatiquement depuis les noms
#         connus des colonnes V0 et V1 de la feuille UPDRSIII.
#         Vous pouvez passer explicitement, ex :
#             ["V12H_APRES", "DEMIVIE_AGONISTE", "H_LDOPA", "DOSE_LDOPA",
#              "OFF_H", "ON_H", "ONOFF_", "OFFON_", "BEST_ON", "SCORE_TOTAL",
#              "DOPASENSIBILITE", "CHAMP14"]
#     prefixes_frequence : liste de préfixes identifiant les colonnes FREQUENCE.
#         Par défaut : ["PLOT", "AMPLITUDE", "UNITE", "DUREE", "FREQUENCE",
#                       "IMPEDANCE", "NVPLOT", "NVAMPLITUDE", "NVUNITE",
#                       "NVDUREE", "NVFREQUENCE", "NVIMPEDANCE"]
#     """
#     visites = ["V0", "VC", "V1", "V3", "V5"]

#     # ------------------------------------------------------------------ #
#     # Préfixes par défaut                                                  #
#     # ------------------------------------------------------------------ #
#     if prefixes_updrs is None:
#         prefixes_updrs = [
#             "V12H_APRES", "DEMIVIE_AGONISTE", "H_LDOPA", "DOSE_LDOPA",
#             "OFF_H", "ON_H", "ONOFF_H", "OFF_H1", "OFFON_H",
#             "ON_H15", "ON_H30", "ON_H45", "ON_H60", "ON_H90", "ON_H120",
#             "ONOFF_", "OFFON_", "OFF_", "ON_",
#             "BEST_ON", "SCORE_TOTAL", "DOPASENSIBILITE",
#             "CHAMP14", "H_DERNPRISE_DOPA", "SS_TOTAL",
#         ]

#     if prefixes_frequence is None:
#         prefixes_frequence = [
#             "PLOT", "AMPLITUDE", "UNITE", "DUREE", "FREQUENCE", "IMPEDANCE",
#             "NVPLOT", "NVAMPLITUDE", "NVUNITE", "NVDUREE",
#             "NVFREQUENCE", "NVIMPEDANCE",
#         ]

#     # ------------------------------------------------------------------ #
#     # Helpers                                                              #
#     # ------------------------------------------------------------------ #
#     def extraire_visite(col):
#         """Retourne (base, visite) ou (col, None)."""
#         for v in sorted(visites, key=len, reverse=True):
#             if col.endswith(f"_{v}"):
#                 return col[:-(len(v) + 1)], v
#         return col, None

#     def base_sans_visite(col):
#         return extraire_visite(col)[0]

#     def visite_col(col):
#         return extraire_visite(col)[1]

#     def est_updrs(col):
#         base = base_sans_visite(col)
#         return any(base.startswith(p) or base == p.rstrip("_")
#                    for p in prefixes_updrs)

#     def est_frequence(col):
#         base = base_sans_visite(col)
#         return any(base.startswith(p) for p in prefixes_frequence)

#     # ------------------------------------------------------------------ #
#     # Classement de chaque colonne (hors SUBJID)                          #
#     # ------------------------------------------------------------------ #
#     cols_normales   = []
#     cols_updrs      = []
#     cols_frequence  = []

#     for col in df.columns:
#         if col == "SUBJID":
#             continue
#         if est_frequence(col):
#             cols_frequence.append(col)
#         elif est_updrs(col):
#             cols_updrs.append(col)
#         else:
#             cols_normales.append(col)

#     # ------------------------------------------------------------------ #
#     # 1. Colonnes normales : grouper par variable, puis par visite        #
#     # ------------------------------------------------------------------ #
#     from collections import defaultdict

#     groupes = defaultdict(dict)   # base -> {visite: col_name}
#     ordre_bases = []
#     seen_bases = set()

#     for col in cols_normales:
#         base, v = extraire_visite(col)
#         if base not in seen_bases:
#             ordre_bases.append(base)
#             seen_bases.add(base)
#         if v:
#             groupes[base][v] = col

#     ordre_normal = []
#     for base in ordre_bases:
#         if groupes[base]:
#             for v in visites:
#                 if v in groupes[base]:
#                     ordre_normal.append(groupes[base][v])
#         else:
#             # colonne sans suffixe visite
#             ordre_normal.append(base)

#     # ------------------------------------------------------------------ #
#     # 2. UPDRSIII : bloc complet par visite (copier-coller)               #
#     # ------------------------------------------------------------------ #
#     ordre_updrs = []
#     for v in visites:
#         suffix = f"_{v}"
#         bloc = [c for c in cols_updrs if c.endswith(suffix)]
#         # Conserver l'ordre d'origine du df_all
#         bloc_ordonne = [c for c in df.columns if c in bloc]
#         ordre_updrs.extend(bloc_ordonne)

#     # ------------------------------------------------------------------ #
#     # 3. FREQUENCE : même logique, à la fin                               #
#     # ------------------------------------------------------------------ #
#     ordre_frequence = []
#     for v in visites:
#         suffix = f"_{v}"
#         bloc = [c for c in cols_frequence if c.endswith(suffix)]
#         bloc_ordonne = [c for c in df.columns if c in bloc]
#         ordre_frequence.extend(bloc_ordonne)

#     # ------------------------------------------------------------------ #
#     # 4. Assemblage final                                                  #
#     # ------------------------------------------------------------------ #
#     colonnes_finales = (
#         ["SUBJID"]
#         + ordre_normal
#         + ordre_updrs
#         + ordre_frequence
#     )

#     # Sécurité : garder uniquement les colonnes qui existent
#     colonnes_finales = [c for c in colonnes_finales if c in df.columns]

#     # Colonnes éventuellement oubliées (ni normal, ni updrs, ni frequence)
#     deja = set(colonnes_finales)
#     oubliees = [c for c in df.columns if c not in deja]
#     if oubliees:
#         print(f"[INFO] {len(oubliees)} colonnes non classifiées ajoutées à la fin : {oubliees[:10]} ...")
#         colonnes_finales += oubliees

#     return df[colonnes_finales]

In [ ]:
def reordonner_colonnes_par_variable(df):
    """
    Réorganise les colonnes du DataFrame fusionné de façon à grouper
    chaque variable ensemble sur toutes les visites.
    
    Ex: poids_V0, poids_V1, poids_V3 | taille_V0, taille_V1 | PDQ39_V0, PDQ39_V1 ...
    """
    import re

    visites = ["V0", "VC", "V1", "V3", "V5"]  # ordre des visites voulu

    # Séparer SUBJID du reste
    autres_cols = [c for c in df.columns if c != "SUBJID"]

    # Extraire le suffixe de visite d'une colonne
    def extraire_base_et_visite(col):
        for v in sorted(visites, key=len, reverse=True):  # tester les plus longs d'abord
            if col.endswith(f"_{v}"):
                base = col[:-(len(v) + 1)]  # retire "_Vx"
                return base, v
        return col, None  # colonne sans suffixe visite

    # Construire un dict : base → {visite: colonne}
    from collections import defaultdict, OrderedDict
    groupes = defaultdict(dict)
    sans_visite = []

    
    for col in autres_cols:
        base, visite = extraire_base_et_visite(col)
        if visite:
            groupes[base][visite] = col
        else:
            sans_visite.append(col)

    # Reconstruire l'ordre des colonnes :
    # Pour chaque base (dans l'ordre d'apparition original), 
    # mettre V0, VC, V1, V3, V5 côte à côte
    ordre_bases = list(dict.fromkeys(
        extraire_base_et_visite(c)[0] for c in autres_cols
        if extraire_base_et_visite(c)[1] is not None
    ))

    colonnes_ordonnees = ["SUBJID"]
    for base in ordre_bases:
        for v in visites:
            if v in groupes[base]:
                colonnes_ordonnees.append(groupes[base][v])

    # Ajouter les colonnes sans suffixe visite à la fin
    colonnes_ordonnees += sans_visite

    # Garder seulement les colonnes qui existent vraiment
    colonnes_ordonnees = [c for c in colonnes_ordonnees if c in df.columns]

    return df[colonnes_ordonnees]

# FUSIONNER 2 

In [56]:
import pandas as pd
from pathlib import Path


def concat_visites(fichiers: dict) -> pd.DataFrame:
    """
    Concatène les fichiers Excel de visites cliniques en un seul DataFrame large.

    Parameters
    ----------
    fichiers : dict
        Dictionnaire {nom_visite: chemin_fichier}, ex:
        {
            'V0': 'data/V0.xlsx',
            'Vc': 'data/Vc.xlsx',
            'V1': 'data/V1.xlsx',
            'V3': 'data/V3.xlsx',
            'V5': 'data/V5.xlsx',
        }

    Returns
    -------
    pd.DataFrame
        DataFrame large avec SUBJID comme index et les colonnes organisées par
        feuille puis par visite : DATE_V0, DATE_Vc, DATE_V1, ..., POIDS_V0, ...
        Les feuilles UPDRS sont concaténées verticalement (colonnes différentes).
        La feuille FREQUENCE (V1, V3, V5 seulement) est placée à la fin.
    """

    # Feuille spéciale avec colonnes différentes selon visite
    FEUILLE_UPDRS = "UPDRS"
    # Feuille disponible uniquement dans V1, V3, V5
    FEUILLE_FREQUENCE = "FREQUENCE"

    # Ordre des visites
    ordre_visites = list(fichiers.keys())

    # ------------------------------------------------------------------ #
    # 1. Lecture de tous les fichiers                                     #
    # ------------------------------------------------------------------ #
    donnees = {}  # {visite: {feuille: df}}
    for visite, chemin in fichiers.items():
        if not Path(chemin).exists():
            print(f"[ATTENTION] Fichier introuvable : {chemin} — visite {visite} ignorée.")
            continue
        sheets = pd.read_excel(chemin, sheet_name=None, dtype=str)
        donnees[visite] = sheets

    # ------------------------------------------------------------------ #
    # 2. Découverte de toutes les feuilles (ordre naturel du 1er fichier) #
    # ------------------------------------------------------------------ #
    premier_visite = next(iter(donnees))
    toutes_feuilles_ordonnees = list(donnees[premier_visite].keys())

    # Sépare feuilles normales / UPDRS / FREQUENCE
    feuilles_normales = [
        f for f in toutes_feuilles_ordonnees
        if f != FEUILLE_UPDRS and f != FEUILLE_FREQUENCE
    ]
    a_updrs = FEUILLE_UPDRS in toutes_feuilles_ordonnees
    a_frequence = FEUILLE_FREQUENCE in toutes_feuilles_ordonnees

    # ------------------------------------------------------------------ #
    # 3. Construction du DataFrame pour les feuilles normales             #
    # ------------------------------------------------------------------ #
    dfs_feuilles = []

    for feuille in feuilles_normales:
        # Collecte les df par visite pour cette feuille
        dfs_visite = {}
        for visite in ordre_visites:
            if visite not in donnees:
                continue
            if feuille not in donnees[visite]:
                continue
            df = donnees[visite][feuille].copy()
            if "SUBJID" not in df.columns:
                print(f"[ATTENTION] SUBJID absent dans {feuille}/{visite}, ignoré.")
                continue
            df = df.set_index("SUBJID")
            # Retire les colonnes méta inutiles (VISITE déjà encodée dans le suffix)
            cols_a_drop = [c for c in ["VISITE"] if c in df.columns]
            df = df.drop(columns=cols_a_drop)
            dfs_visite[visite] = df

        if not dfs_visite:
            continue

        # Colonnes communes à toutes les visites présentes (même ordre que 1ère visite)
        ref_cols = list(next(iter(dfs_visite.values())).columns)

        # Construit un df large : pour chaque colonne, met côte à côte les visites
        morceaux = []
        for col in ref_cols:
            for visite, df_v in dfs_visite.items():
                if col in df_v.columns:
                    s = df_v[[col]].rename(columns={col: f"{col}_{visite}"})
                    morceaux.append(s)

        df_feuille = pd.concat(morceaux, axis=1)
        dfs_feuilles.append(df_feuille)

    # Jointure de toutes les feuilles normales
    if dfs_feuilles:
        df_final = dfs_feuilles[0]
        for df_f in dfs_feuilles[1:]:
            df_final = df_final.join(df_f, how="outer")
    else:
        df_final = pd.DataFrame()

    # ------------------------------------------------------------------ #
    # 4. Feuille UPDRS — colonnes différentes : concat verticale          #
    # ------------------------------------------------------------------ #
    if a_updrs:
        blocs_updrs = []
        for visite in ordre_visites:
            if visite not in donnees:
                continue
            if FEUILLE_UPDRS not in donnees[visite]:
                continue
            df = donnees[visite][FEUILLE_UPDRS].copy()
            if "SUBJID" not in df.columns:
                continue
            df = df.set_index("SUBJID")
            cols_a_drop = [c for c in ["VISITE"] if c in df.columns]
            df = df.drop(columns=cols_a_drop)
            # Suffixe toutes les colonnes avec la visite
            df.columns = [f"{c}_{visite}" for c in df.columns]
            blocs_updrs.append(df)

        if blocs_updrs:
            df_updrs = pd.concat(blocs_updrs, axis=1)
            df_final = df_final.join(df_updrs, how="outer") if not df_final.empty else df_updrs

    # ------------------------------------------------------------------ #
    # 5. Feuille FREQUENCE — uniquement V1, V3, V5 — placée à la fin     #
    # ------------------------------------------------------------------ #
    if a_frequence:
        visites_freq = [v for v in ["V1", "V3", "V5"] if v in ordre_visites]
        dfs_freq_visite = {}
        for visite in visites_freq:
            if visite not in donnees:
                continue
            if FEUILLE_FREQUENCE not in donnees[visite]:
                continue
            df = donnees[visite][FEUILLE_FREQUENCE].copy()
            if "SUBJID" not in df.columns:
                continue
            df = df.set_index("SUBJID")
            cols_a_drop = [c for c in ["VISITE"] if c in df.columns]
            df = df.drop(columns=cols_a_drop)
            dfs_freq_visite[visite] = df

        if dfs_freq_visite:
            ref_cols_freq = list(next(iter(dfs_freq_visite.values())).columns)
            morceaux_freq = []
            for col in ref_cols_freq:
                for visite, df_v in dfs_freq_visite.items():
                    if col in df_v.columns:
                        s = df_v[[col]].rename(columns={col: f"{col}_{visite}"})
                        morceaux_freq.append(s)

            df_frequence = pd.concat(morceaux_freq, axis=1)
            df_final = df_final.join(df_frequence, how="outer") if not df_final.empty else df_frequence

    df_final.index.name = "SUBJID"
    return df_final.reset_index()



In [60]:

# --------------------------------------------------------------------------- #
# Exemple d'utilisation                                                        #
# --------------------------------------------------------------------------- #
# if __name__ == "__main__":
fichiers = {
    "V0": "Output/version_4/V0.xlsx",
    "Vc": "Output/version_4/Vc.xlsx",
    "V1": "Output/version_4/V1.xlsx",
    "V3": "Output/version_4/V3.xlsx",
    "V5": "Output/version_4/V5.xlsx",
}

df = concat_visites(fichiers)
print(df.shape)
print(df.columns.tolist())
print(df.head())

# Sauvegarde optionnelle
# df.to_excel("resultat_concatene.xlsx", index=False)

[OK] V0 -> feuilles : ['V0', 'PDQ39', 'UPDRSIII', 'UPDRSIV', 'ECMP', 'QUIP', 'LARS', 'UPPS', 'HAMD', 'HAMA', 'MOCA', 'DIGITSMT_TRAILMT_DKEFS', 'PSYCHOTROPES', 'LEDD']
[OK] Vc -> feuilles : ['Vc', 'PSYCHOTROPES', 'LEDD']
[OK] V1 -> feuilles : ['V1', 'PDQ39', 'UPDRSIII', 'UPDRSIV', 'ECMP', 'QUIP', 'LARS', 'UPPS', 'HAMD', 'HAMA', 'MOCA', 'DIGITSMT_TRAILMT_DKEFS', 'PSYCHOTROPES', 'LEDD', 'FREQUENCE']
[OK] V3 -> feuilles : ['V3', 'PDQ39', 'UPDRSIII', 'UPDRSIV', 'ECMP', 'QUIP', 'LARS', 'UPPS', 'HAMD', 'HAMA', 'MOCA', 'DIGITSMT_TRAILMT_DKEFS', 'PSYCHOTROPES', 'LEDD', 'FREQUENCE']
[OK] V5 -> feuilles : ['V5', 'PDQ39', 'UPDRSIII', 'UPDRSIV', 'ECMP', 'QUIP', 'LARS', 'UPPS', 'HAMD', 'HAMA', 'MOCA', 'DIGITSMT_TRAILMT_DKEFS', 'PSYCHOTROPES', 'LEDD']

Feuilles normales : ['V0', 'PDQ39', 'UPDRSIV', 'ECMP', 'QUIP', 'LARS', 'UPPS', 'HAMD', 'HAMA', 'MOCA', 'DIGITSMT_TRAILMT_DKEFS', 'PSYCHOTROPES', 'LEDD', 'Vc', 'V1', 'V3', 'V5']
UPDRS present    : True  (UPDRSIII)
FREQUENCE present: True

(835, 1085)
['

In [62]:
df.to_excel("data03.xlsx",index=False)

In [59]:
import pandas as pd
from pathlib import Path


def concat_visites(fichiers: dict) -> pd.DataFrame:
    """
    Concatène les fichiers Excel de visites cliniques en un seul DataFrame large.

    Parameters
    ----------
    fichiers : dict
        Dictionnaire ORDONNÉ {nom_visite: chemin_fichier}, ex:
        {
            'V0': 'data/V0.xlsx',
            'Vc': 'data/Vc.xlsx',
            'V1': 'data/V1.xlsx',
            'V3': 'data/V3.xlsx',
            'V5': 'data/V5.xlsx',
        }

    Returns
    -------
    pd.DataFrame
        DataFrame large avec SUBJID comme clé, colonnes organisées par feuille
        puis par visite. UPDRSIII = copier-coller côte à côte. FREQUENCE à la fin.
    """

    FEUILLE_UPDRS     = "UPDRSIII"   # <-- adaptez si le nom diffère
    FEUILLE_FREQUENCE = "FREQUENCE"

    ordre_visites = list(fichiers.keys())

    # ------------------------------------------------------------------ #
    # 1. Lecture de tous les fichiers                                     #
    # ------------------------------------------------------------------ #
    donnees = {}  # {visite: {feuille: df}}
    for visite, chemin in fichiers.items():
        if not Path(chemin).exists():
            print(f"[ATTENTION] Fichier introuvable : {chemin} — visite {visite} ignorée.")
            continue
        sheets = pd.read_excel(chemin, sheet_name=None, dtype=str)
        donnees[visite] = sheets
        print(f"[OK] {visite} -> feuilles : {list(sheets.keys())}")

    # ------------------------------------------------------------------ #
    # 2. Ordre global des feuilles = union dans l'ordre de première       #
    #    apparition en parcourant les visites dans l'ordre                #
    # ------------------------------------------------------------------ #
    ordre_feuilles_global = []
    seen = set()
    for visite in ordre_visites:
        if visite not in donnees:
            continue
        for feuille in donnees[visite].keys():
            if feuille not in seen:
                ordre_feuilles_global.append(feuille)
                seen.add(feuille)

    feuilles_normales = [
        f for f in ordre_feuilles_global
        if f != FEUILLE_UPDRS and f != FEUILLE_FREQUENCE
    ]
    a_updrs     = FEUILLE_UPDRS     in seen
    a_frequence = FEUILLE_FREQUENCE in seen

    print(f"\nFeuilles normales : {feuilles_normales}")
    print(f"UPDRS present    : {a_updrs}  ({FEUILLE_UPDRS})")
    print(f"FREQUENCE present: {a_frequence}\n")

    # ------------------------------------------------------------------ #
    # Helper : lire + nettoyer une feuille d'une visite                  #
    # ------------------------------------------------------------------ #
    def lire_feuille(visite, feuille):
        if visite not in donnees:
            return None
        if feuille not in donnees[visite]:
            return None
        df = donnees[visite][feuille].copy()
        if "SUBJID" not in df.columns:
            print(f"  [ATTENTION] SUBJID absent dans {feuille}/{visite}")
            return None
        df = df.set_index("SUBJID")
        if "VISITE" in df.columns:
            df = df.drop(columns=["VISITE"])
        return df

    # ------------------------------------------------------------------ #
    # 3. Feuilles normales : col1_V0, col1_Vc, col1_V1 ... col2_V0 ...  #
    # ------------------------------------------------------------------ #
    blocs_normaux = []

    for feuille in feuilles_normales:
        dfs_par_visite = {}
        for visite in ordre_visites:
            df = lire_feuille(visite, feuille)
            if df is not None:
                dfs_par_visite[visite] = df

        if not dfs_par_visite:
            continue

        # Colonnes de référence = union dans l'ordre de première apparition
        ref_cols = []
        seen_cols = set()
        for df_v in dfs_par_visite.values():
            for c in df_v.columns:
                if c not in seen_cols:
                    ref_cols.append(c)
                    seen_cols.add(c)

        # Pour chaque colonne, colle les visites côte à côte
        morceaux = []
        for col in ref_cols:
            for visite, df_v in dfs_par_visite.items():
                if col in df_v.columns:
                    s = df_v[[col]].rename(columns={col: f"{col}_{visite}"})
                    morceaux.append(s)

        if morceaux:
            blocs_normaux.append(pd.concat(morceaux, axis=1))

    # ------------------------------------------------------------------ #
    # 4. UPDRSIII : copier-coller brut cote a cote (toutes colonnes)     #
    # ------------------------------------------------------------------ #
    bloc_updrs = None
    if a_updrs:
        morceaux_updrs = []
        for visite in ordre_visites:
            df = lire_feuille(visite, FEUILLE_UPDRS)
            if df is None:
                continue
            df.columns = [f"{c}_{visite}" for c in df.columns]
            morceaux_updrs.append(df)

        if morceaux_updrs:
            bloc_updrs = pd.concat(morceaux_updrs, axis=1)

    # ------------------------------------------------------------------ #
    # 5. FREQUENCE : meme logique que feuilles normales, mais a la fin   #
    # ------------------------------------------------------------------ #
    bloc_frequence = None
    if a_frequence:
        dfs_freq = {}
        for visite in ordre_visites:
            df = lire_feuille(visite, FEUILLE_FREQUENCE)
            if df is not None:
                dfs_freq[visite] = df

        if dfs_freq:
            ref_cols_freq = []
            seen_cols_freq = set()
            for df_v in dfs_freq.values():
                for c in df_v.columns:
                    if c not in seen_cols_freq:
                        ref_cols_freq.append(c)
                        seen_cols_freq.add(c)

            morceaux_freq = []
            for col in ref_cols_freq:
                for visite, df_v in dfs_freq.items():
                    if col in df_v.columns:
                        s = df_v[[col]].rename(columns={col: f"{col}_{visite}"})
                        morceaux_freq.append(s)

            if morceaux_freq:
                bloc_frequence = pd.concat(morceaux_freq, axis=1)

    # ------------------------------------------------------------------ #
    # 6. Assemblage final                                                 #
    # ------------------------------------------------------------------ #
    tous_blocs = blocs_normaux.copy()
    if bloc_updrs is not None:
        tous_blocs.append(bloc_updrs)
    if bloc_frequence is not None:
        tous_blocs.append(bloc_frequence)

    if not tous_blocs:
        print("[ERREUR] Aucun bloc construit — verifiez les fichiers.")
        return pd.DataFrame()

    df_final = tous_blocs[0]
    for bloc in tous_blocs[1:]:
        df_final = df_final.join(bloc, how="outer")

    df_final.index.name = "SUBJID"
    return df_final.reset_index()


# --------------------------------------------------------------------------- #
# Exemple d'utilisation                                                        #
# --------------------------------------------------------------------------- #
if __name__ == "__main__":
    fichiers = {
        "V0": "V0.xlsx",
        "Vc": "Vc.xlsx",
        "V1": "V1.xlsx",
        "V3": "V3.xlsx",
        "V5": "V5.xlsx",
    }

    df = concat_visites(fichiers)

    print(f"\nShape final : {df.shape}")
    print("Colonnes :", df.columns.tolist())
    print(df.head())

    # Sauvegarde
    # df.to_excel("resultat_concatene.xlsx", index=False)

[ATTENTION] Fichier introuvable : V0.xlsx — visite V0 ignorée.
[ATTENTION] Fichier introuvable : Vc.xlsx — visite Vc ignorée.
[ATTENTION] Fichier introuvable : V1.xlsx — visite V1 ignorée.
[ATTENTION] Fichier introuvable : V3.xlsx — visite V3 ignorée.
[ATTENTION] Fichier introuvable : V5.xlsx — visite V5 ignorée.

Feuilles normales : []
UPDRS present    : False  (UPDRSIII)
FREQUENCE present: False

[ERREUR] Aucun bloc construit — verifiez les fichiers.

Shape final : (0, 0)
Colonnes : []
Empty DataFrame
Columns: []
Index: []
